[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_05/notebook_5_1_classification_metrics.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 5.1: Classification Metrics Deep Dive

**Chapter 5: Model Evaluation**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Calculate and interpret all metrics from a confusion matrix
2. Understand the sensitivity-specificity trade-off
3. Calculate PPV and NPV as functions of prevalence
4. Select appropriate metrics for different clinical scenarios
5. Recognize when accuracy is misleading

## Clinical Context

**Choosing the right metric is critical**:
- **Screening tests**: High sensitivity (don't miss cases)
- **Confirmatory tests**: High specificity (avoid false alarms)
- **Resource allocation**: PPV matters (how many positives are true?)
- **Rare diseases**: Accuracy is misleading with class imbalance

**Real consequences**:
- Low sensitivity → Missed diagnoses → Patient harm
- Low specificity → False positives → Unnecessary procedures
- Low PPV → Alert fatigue → Clinicians ignore warnings

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported")

## 1. The Confusion Matrix Foundation

In [ ]:
# Generate simple example
# Screening test for rare disease (3% prevalence)
np.random.seed(42)
n_patients = 1000
prevalence = 0.03

# True labels
y_true = np.random.choice([0, 1], n_patients, p=[1-prevalence, prevalence])

# Simulated predictions (80% sensitivity, 95% specificity)
y_pred = y_true.copy()

# False negatives (missed 20% of positives)
positive_idx = np.where(y_true == 1)[0]
fn_idx = np.random.choice(positive_idx, size=int(0.20 * len(positive_idx)), replace=False)
y_pred[fn_idx] = 0

# False positives (5% of negatives)
negative_idx = np.where(y_true == 0)[0]
fp_idx = np.random.choice(negative_idx, size=int(0.05 * len(negative_idx)), replace=False)
y_pred[fp_idx] = 1

# Calculate confusion matrix
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("="*70)
print("CONFUSION MATRIX")
print("="*70)
print(f"\nTotal patients: {n_patients}")
print(f"True prevalence: {y_true.sum()} / {n_patients} = {y_true.mean():.1%}")
print(f"\nConfusion Matrix:")
print(f"                 Predicted Negative  Predicted Positive")
print(f"Actually Negative      {tn:4d} (TN)        {fp:4d} (FP)")
print(f"Actually Positive      {fn:4d} (FN)        {tp:4d} (TP)")
print("\n" + "="*70)

In [ ]:
# Visualize confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Counts
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Predicted Neg', 'Predicted Pos'],
            yticklabels=['Actually Neg', 'Actually Pos'],
            cbar=False, annot_kws={'size': 16})
ax.set_title('Confusion Matrix (Counts)', fontweight='bold', fontsize=14)

# Percentages
ax = axes[1]
cm_pct = cm.astype('float') / cm.sum() * 100
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            xticklabels=['Predicted Neg', 'Predicted Pos'],
            yticklabels=['Actually Neg', 'Actually Pos'],
            cbar=False, annot_kws={'size': 16})
ax.set_title('Confusion Matrix (Percentages)', fontweight='bold', fontsize=14)

plt.tight_layout()
plt.show()

## 2. Core Metrics: Sensitivity and Specificity

In [ ]:
# Calculate all metrics manually
def calculate_all_metrics(tn, fp, fn, tp):
    """
    Calculate all classification metrics from confusion matrix.
    """
    # Basic counts
    total = tn + fp + fn + tp
    actual_positive = tp + fn
    actual_negative = tn + fp
    predicted_positive = tp + fp
    predicted_negative = tn + fn

    # Metrics
    accuracy = (tp + tn) / total

    # Sensitivity (Recall, True Positive Rate)
    sensitivity = tp / actual_positive if actual_positive > 0 else 0

    # Specificity (True Negative Rate)
    specificity = tn / actual_negative if actual_negative > 0 else 0

    # Positive Predictive Value (Precision)
    ppv = tp / predicted_positive if predicted_positive > 0 else 0

    # Negative Predictive Value
    npv = tn / predicted_negative if predicted_negative > 0 else 0

    # F1 Score
    f1 = 2 * (ppv * sensitivity) / (ppv + sensitivity) if (ppv + sensitivity) > 0 else 0

    # False Positive Rate
    fpr = fp / actual_negative if actual_negative > 0 else 0

    # False Negative Rate
    fnr = fn / actual_positive if actual_positive > 0 else 0

    return {
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ppv': ppv,
        'npv': npv,
        'f1': f1,
        'fpr': fpr,
        'fnr': fnr
    }

metrics = calculate_all_metrics(tn, fp, fn, tp)

print("="*70)
print("ALL CLASSIFICATION METRICS")
print("="*70)
print(f"\nACCURACY: {metrics['accuracy']:.1%}")
print(f"  (TP + TN) / Total = ({tp} + {tn}) / {n_patients}")
print(f"  ⚠️ Can be misleading with imbalance!")

print(f"\nSENSITIVITY (Recall, TPR): {metrics['sensitivity']:.1%}")
print(f"  TP / (TP + FN) = {tp} / ({tp} + {fn})")
print(f"  How many actual positives did we catch?")

print(f"\nSPECIFICITY (TNR): {metrics['specificity']:.1%}")
print(f"  TN / (TN + FP) = {tn} / ({tn} + {fp})")
print(f"  How many actual negatives did we correctly identify?")

print(f"\nPOSITIVE PREDICTIVE VALUE (Precision): {metrics['ppv']:.1%}")
print(f"  TP / (TP + FP) = {tp} / ({tp} + {fp})")
print(f"  Of predicted positives, how many are truly positive?")

print(f"\nNEGATIVE PREDICTIVE VALUE: {metrics['npv']:.1%}")
print(f"  TN / (TN + FN) = {tn} / ({tn} + {fn})")
print(f"  Of predicted negatives, how many are truly negative?")

print(f"\nF1 SCORE: {metrics['f1']:.3f}")
print(f"  2 * (PPV * Sensitivity) / (PPV + Sensitivity)")
print(f"  Harmonic mean of precision and recall")

print(f"\nFALSE POSITIVE RATE: {metrics['fpr']:.1%}")
print(f"  FP / (FP + TN) = {fp} / ({fp} + {tn})")
print(f"  1 - Specificity")

print(f"\nFALSE NEGATIVE RATE: {metrics['fnr']:.1%}")
print(f"  FN / (FN + TP) = {fn} / ({fn} + {tp})")
print(f"  1 - Sensitivity")
print("\n" + "="*70)

## 3. PPV Depends on Prevalence!

In [ ]:
def calculate_ppv_npv(sensitivity, specificity, prevalence):
    """
    Calculate PPV and NPV given test characteristics and prevalence.

    This is CRITICAL: PPV changes with prevalence even if test is the same!
    """
    # For a population of 1000
    n = 1000

    # True positives and negatives
    actual_pos = n * prevalence
    actual_neg = n * (1 - prevalence)

    # Test results
    tp = actual_pos * sensitivity
    fn = actual_pos * (1 - sensitivity)
    tn = actual_neg * specificity
    fp = actual_neg * (1 - specificity)

    # Predictive values
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    npv = tn / (tn + fn) if (tn + fn) > 0 else 0

    return ppv, npv, tp, fp, tn, fn

# Fixed test: 80% sensitivity, 95% specificity
test_sens = 0.80
test_spec = 0.95

# Vary prevalence
prevalences = [0.001, 0.01, 0.03, 0.10, 0.30, 0.50]
results = []

for prev in prevalences:
    ppv, npv, tp, fp, tn, fn = calculate_ppv_npv(test_sens, test_spec, prev)
    fp_ratio = fp / tp if tp > 0 else float('inf')
    results.append({
        'prevalence': prev,
        'ppv': ppv,
        'npv': npv,
        'fp_per_tp': fp_ratio
    })

results_df = pd.DataFrame(results)

print("="*70)
print(f"PPV AS FUNCTION OF PREVALENCE")
print(f"(Fixed: Sensitivity={test_sens:.0%}, Specificity={test_spec:.0%})")
print("="*70)
print(f"\n{'Prevalence':>12} {'PPV':>10} {'NPV':>10} {'FP per TP':>15}")
print("-" * 70)
for _, row in results_df.iterrows():
    print(f"{row['prevalence']:>11.1%} {row['ppv']:>10.1%} {row['npv']:>10.1%} {row['fp_per_tp']:>15.1f}")

print("\n" + "="*70)
print("💡 KEY INSIGHT: At 0.1% prevalence, 95% specificity yields 20 FP per TP!")
print("   Even 'good' tests perform poorly in screening low-prevalence conditions.")
print("="*70)

In [ ]:
# Visualize PPV vs prevalence
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PPV curve
ax = axes[0]
prevalence_range = np.linspace(0.001, 0.5, 100)
ppv_values = [calculate_ppv_npv(test_sens, test_spec, p)[0] for p in prevalence_range]

ax.plot(prevalence_range * 100, np.array(ppv_values) * 100, linewidth=3, color='steelblue')
ax.axhline(50, color='red', linestyle='--', alpha=0.5, label='PPV = 50%')
ax.set_xlabel('Prevalence (%)', fontsize=12)
ax.set_ylabel('Positive Predictive Value (%)', fontsize=12)
ax.set_title(f'PPV vs Prevalence\n(Sens={test_sens:.0%}, Spec={test_spec:.0%})', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()

# FP per TP ratio
ax = axes[1]
fp_per_tp_values = [calculate_ppv_npv(test_sens, test_spec, p)[2] / calculate_ppv_npv(test_sens, test_spec, p)[3]
                    if calculate_ppv_npv(test_sens, test_spec, p)[3] > 0 else 0
                    for p in prevalence_range]

ax.plot(prevalence_range * 100, fp_per_tp_values, linewidth=3, color='coral')
ax.axhline(1, color='red', linestyle='--', alpha=0.5, label='1 FP per TP')
ax.set_xlabel('Prevalence (%)', fontsize=12)
ax.set_ylabel('False Positives per True Positive', fontsize=12)
ax.set_title('False Positive Burden vs Prevalence', fontweight='bold')
ax.set_ylim([0, 50])
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

## 4. Why Accuracy is Misleading

In [ ]:
# Demonstrate accuracy paradox
print("="*70)
print("THE ACCURACY PARADOX")
print("="*70)

# Scenario: Rare disease (1% prevalence)
n = 1000
prevalence = 0.01
actual_pos = int(n * prevalence)  # 10 cases
actual_neg = n - actual_pos  # 990 cases

print(f"\nScenario: Screening for rare disease (prevalence = {prevalence:.1%})")
print(f"Population: {n} patients ({actual_pos} positive, {actual_neg} negative)\n")

# Model 1: Always predict negative (lazy baseline)
print("Model 1: ALWAYS PREDICT NEGATIVE")
tp1, fp1, tn1, fn1 = 0, 0, actual_neg, actual_pos
acc1 = (tp1 + tn1) / n
sens1 = tp1 / actual_pos if actual_pos > 0 else 0
spec1 = tn1 / actual_neg if actual_neg > 0 else 0
ppv1 = 0  # No positive predictions

print(f"  Accuracy: {acc1:.1%} 🎉 (Looks great!)")
print(f"  Sensitivity: {sens1:.1%} ❌ (Catches nothing!)")
print(f"  Specificity: {spec1:.1%}")
print(f"  PPV: N/A (no positive predictions)")
print(f"  Clinical utility: ZERO - misses ALL cases!\n")

# Model 2: Reasonable classifier (80% sens, 95% spec)
print("Model 2: REASONABLE CLASSIFIER")
tp2 = int(actual_pos * 0.80)
fn2 = actual_pos - tp2
tn2 = int(actual_neg * 0.95)
fp2 = actual_neg - tn2

acc2 = (tp2 + tn2) / n
sens2 = tp2 / actual_pos
spec2 = tn2 / actual_neg
ppv2 = tp2 / (tp2 + fp2) if (tp2 + fp2) > 0 else 0

print(f"  Accuracy: {acc2:.1%} (Slightly worse than baseline!)")
print(f"  Sensitivity: {sens2:.1%} ✓ (Catches most cases)")
print(f"  Specificity: {spec2:.1%} ✓ (Few false alarms)")
print(f"  PPV: {ppv2:.1%} ({fp2/tp2:.0f} FP per TP)")
print(f"  Clinical utility: HIGH - detects {tp2}/{actual_pos} cases")

print("\n" + "="*70)
print("💡 Model 1 has HIGHER accuracy but is CLINICALLY USELESS!")
print("   Accuracy is misleading when classes are imbalanced.")
print("="*70)

## 5. Sensitivity-Specificity Trade-off

In [ ]:
# Generate predictions with probabilities
np.random.seed(42)
n = 1000
y_true = np.random.choice([0, 1], n, p=[0.7, 0.3])

# Simulate model probabilities
# Positives have higher scores on average
y_prob = np.where(y_true == 1,
                  np.random.beta(8, 2, n),  # Positives: skewed high
                  np.random.beta(2, 8, n))  # Negatives: skewed low

# Test different thresholds
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
threshold_results = []

for thresh in thresholds:
    y_pred = (y_prob >= thresh).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0

    threshold_results.append({
        'threshold': thresh,
        'sensitivity': sens,
        'specificity': spec,
        'ppv': ppv,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn
    })

results_df = pd.DataFrame(threshold_results)

print("="*70)
print("SENSITIVITY-SPECIFICITY TRADE-OFF")
print("="*70)
print(f"\n{'Threshold':>12} {'Sensitivity':>12} {'Specificity':>12} {'PPV':>10} {'TP':>6} {'FP':>6}")
print("-" * 70)
for _, row in results_df.iterrows():
    print(f"{row['threshold']:>12.1f} {row['sensitivity']:>12.1%} {row['specificity']:>12.1%} "
          f"{row['ppv']:>10.1%} {row['tp']:>6.0f} {row['fp']:>6.0f}")

print("\n" + "="*70)
print("💡 Lower threshold → Higher sensitivity but lower specificity (more FP)")
print("   Higher threshold → Higher specificity but lower sensitivity (more FN)")
print("   Choose based on clinical cost of FP vs FN!")
print("="*70)

In [ ]:
# Visualize trade-off
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(results_df['threshold'], results_df['sensitivity'],
        'o-', linewidth=3, markersize=8, label='Sensitivity', color='steelblue')
ax.plot(results_df['threshold'], results_df['specificity'],
        's-', linewidth=3, markersize=8, label='Specificity', color='coral')
ax.plot(results_df['threshold'], results_df['ppv'],
        '^-', linewidth=3, markersize=8, label='PPV', color='green')

ax.axvline(0.5, color='red', linestyle='--', alpha=0.3, label='Default threshold (0.5)')
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('Sensitivity-Specificity-PPV Trade-off', fontweight='bold', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

## 6. Choosing the Right Metric

In [ ]:
print("="*70)
print("METRIC SELECTION GUIDE")
print("="*70)

print("\n1. SCREENING TESTS (e.g., cancer screening)")
print("   Goal: Don't miss any cases")
print("   Primary metric: SENSITIVITY (aim for >95%)")
print("   Accept: Lower specificity (more false positives tolerable)")
print("   Example: Mammography for breast cancer")

print("\n2. CONFIRMATORY TESTS (e.g., biopsy)")
print("   Goal: Avoid unnecessary procedures")
print("   Primary metric: SPECIFICITY & PPV (aim for >95%)")
print("   Accept: Lower sensitivity (already pre-screened)")
print("   Example: Genetic test for rare mutation")

print("\n3. RESOURCE ALLOCATION (e.g., ICU bed assignment)")
print("   Goal: Efficiently use limited resources")
print("   Primary metric: PPV (precision)")
print("   Reason: Can't follow up on many false positives")
print("   Example: Sepsis alerts in EMR")

print("\n4. SAFETY-CRITICAL (e.g., autonomous systems)")
print("   Goal: Minimize harm from both FP and FN")
print("   Primary metric: F1 score (balance)")
print("   Consider: Cost-weighted metrics")
print("   Example: Drug-drug interaction alerts")

print("\n5. RARE OUTCOMES (e.g., adverse events)")
print("   Goal: Don't be fooled by accuracy")
print("   Primary metric: AUROC, sensitivity at fixed specificity")
print("   Avoid: Accuracy (will be ~99% even for useless model)")
print("   Example: Predicting 1% mortality rate")

print("\n" + "="*70)

## 7. Key Takeaways

### What We Learned

1. **Confusion Matrix**: Foundation for all metrics
   - TP, TN, FP, FN define everything
   - Always visualize to understand model behavior

2. **Sensitivity vs Specificity**:
   - Sensitivity: Fraction of positives caught (recall)
   - Specificity: Fraction of negatives correctly identified
   - Inverse relationship via threshold

3. **PPV Depends on Prevalence**:
   - Same test performs differently in different populations
   - Low prevalence → Low PPV even with high specificity
   - Critical for screening programs

4. **Accuracy is Misleading**:
   - High accuracy with class imbalance is meaningless
   - "Always predict negative" achieves 99% accuracy for 1% prevalence
   - Use sensitivity/specificity instead

5. **Threshold Selection**:
   - Default 0.5 is arbitrary
   - Choose based on clinical cost of errors
   - Lower for screening, higher for confirmatory

### Clinical Considerations

- **False positives**: Anxiety, unnecessary procedures, cost
- **False negatives**: Missed diagnoses, delayed treatment, harm
- **Prevalence shifts**: Model trained on 10% prevalence may fail at 1%
- **Alert fatigue**: Low PPV → Clinicians ignore alerts

### Connection to Journey Stories

**Marcus (Sepsis)**: High sensitivity (90%) but FP rate causes alert fatigue (see Notebook 7.3)

**Yuki (AFib)**: 0.1% prevalence → Even 98% specificity yields terrible PPV (see Notebook 7.4)

**Jamal (Nodules)**: 95% sensitivity → 2.5 FP per image → 250 FP per scan (see Notebook 6.5)

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 5: Model Evaluation)*

## Exercises

1. **Prevalence Sensitivity**: Calculate how PPV changes when prevalence drops from 10% to 1% for a test with 90% sensitivity and 95% specificity.

2. **Threshold Optimization**: Given clinical costs (FP = $100, FN = $10,000), find optimal threshold.

3. **Multi-class Metrics**: Extend confusion matrix to 3-class problem (e.g., no disease, mild, severe).

4. **Stratified Evaluation**: Calculate metrics separately for different age groups. When do they differ?

5. **Real Data**: Apply to public dataset (MIMIC, eICU). Compare metrics across different hospitals.